## Wstęp do Uczenia Maszynowego

#### Laboratorium 02


### Zakres laboratorium

- drzewa decyzyjne w regresji i klasyfikacji
- kryteria podziału oraz złożoność drzewa
- macierz pomyłek i podstawowe metryki klasyfikacji


In [3]:
# Import potrzebnych pakietów
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn import tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score


### 0. Drzewa decyzyjne - regresja

Na chwilę wrócimy do zajęć lab 01.

Pracowaliśmy ostatnio nad zadaniem regresji i przewidywaniem wynagrodzenia dla zawodników baseballu.

Stworzyliśmy pierwszy model i ocenialiśmy jego jakość mierząc błąd średniokwadratowy.

In [4]:
## Kod pobierający dane
Hitters = pd.read_csv("https://raw.githubusercontent.com/kozaka93/2026Z-MachineLearning/refs/heads/main/labs/lab01/Hitters.csv", index_col =[0])

## Ograniczamy zbiór do trzech zmiennych
Hitters_small = Hitters[["Years", "Hits", "Salary"]]

## Usuwamy braki danych
Hitters_small = Hitters_small.dropna()

## Dzielimy zbiór danych na y (zmienną docelową) i X (zmienne wejściowe)
y = Hitters_small.Salary
X = Hitters_small.drop(["Salary"], axis=1) 

# Podział na zbiór treningowy (do uczenia) i testowy (do końcowej oceny)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=123
)

# Wydzielenie zbioru walidacyjnego ze zbioru treningowego
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=456
)

## Budujemy drzewo z 3 liśćmi (regresja).
Tree3 = tree.DecisionTreeRegressor(max_leaf_nodes=3, random_state=123)
Tree3 = Tree3.fit(X_train, y_train)

In [ ]:
## Rysunek drzewa
tree.plot_tree(Tree3, 
               feature_names=Tree3.feature_names_in_.tolist(),
               filled=True)
plt.show()

### Regiony 

<img src="Five-Region-Examples.png" width="400"/>

###### Źródło:  James, G., Witten, D., Hastie, T., Tibshirani, R., Taylor, J. (2023). An Introduction to Statistical Learning with Applications in Python, Springer Science+Business Media, New York. https://www.statlearning.com/

-----
##### *Zadanie 1*
Na podstawie informacji o utworzonym drzewie zdefiniuj regiony $R_1$, $R_2$ i $R_3$.

-----
##### *Zadanie 2*
Na bazie przedstawionych poniżej regionów narysuj odpowiadające im drzewo decyzyjne.

<img src="Regions_Task.png" alt="drawing" width="400"/>

#### Jak konstruować regiony w przypadku drzewa dla zadania regresji?
**Cel**: Znaleźć $R_1, R_2, \dots, R_J$, które minimalizują $RSS$.

$$RSS = \sum_{j = 1}^{J} \sum_{i \in R_j}^{}(y_i - \hat{y}_{R_j})^2$$

Niestety nie jesteśmy w stanie rozpatrzeć wszystkich możliwości - zbyt złożone obliczeniowo.

Używamy podejścia *top-down, greedy*:
- *top-down* — zaczynamy od wszystkich obserwacji w jednym regionie, a następnie w kolejnych krokach dzielimy go na mniejsze regiony
- *greedy* - na każdym etapie procesu budowania drzewa dokonywany jest najlepszy podział, nie patrzymy w przyszłość

Proces powtarzamy do momentu spełnienia kryterium stopu, np. w liściu nie może być mniej niż 5 obserwacji.



-----
##### *Zadanie 3*
Przeprowadź eksperyment pokazujący wartość miary MSE na zbiorze uczącym i walidacyjnym w zależności od wartości parametru `max_leaf_nodes` — dla wartości od 2 do 30. Zbiór testowy wykorzystaj wyłącznie do końcowej oceny wybranego modelu.

> `max_leaf_nodes` - Grow a tree with max_leaf_nodes in best-first fashion. Best nodes are defined as relative reduction in impurity. If None then unlimited number of leaf nodes (default=None).

Narysuj wykres przedstawiający wyniki eksperymentu. Jaka jest optymalna wartość parametru `max_leaf_nodes`? 

Wytrenuj wybrany model ponownie na całym zbiorze treningowym i oblicz jego MSE na zbiorze testowym.

⚠️ Ze względu na niewielki rozmiar zbioru pojedynczy podział walidacyjny może dawać niestabilne wyniki. W laboratorium 3 zastąpimy go kroswalidacją.

In [ ]:
from sklearn.metrics import mean_squared_error

mse_fit = []
mse_val = []

### 1. Drzewa decyzyjne - klasyfikacja

In [ ]:
pima = pd.read_csv("pima.csv")
# pima = pd.read_csv("link do raw GitHub")

Więcej o danych: https://www.openml.org/search?type=data&sort=runs&id=43483&status=active

### Kryterium podziału
a) wskaźnik błędu klasyfikacji (*classification error rate*) = udział obserwacji ze zbioru treningowego w danym regionie, które nie należą do przeważającej klasy.
    
$$E = 1 - max_{k}(\hat{p}_{mk}) $$
 
$\hat{p}_{mk}$ - proporcja obserwacji zbioru treningowego w m-tym regionie pochodząca z k-tej klasy
   
b) indeks Giniego   

 $$G = \sum_{k=1}^{K}\hat{p}_{mk}(1-\hat{p}_{mk})$$
 
 lub

 $$G = 1 - \sum_{k=1}^{K}\hat{p}_{mk}^2$$

Indeks Giniego przyjmuje małe wartości jeżeli $\hat{p}_{mk}$ jest bliski 0 lub 1. Z tego powodu indeks Giniego jest określany jako miara czystości węzła (*node purity*) - mała wartość wskazuje, że węzeł zawiera głównie obserwacje z jednej klasy.

c) entropia
 
 $$D = - \sum_{k=1}^{K}\hat{p}_{mk}log\hat{p}_{mk}$$

 $0 \leq \hat{p}_{mk} \leq 1 \ \rightarrow 0 \leq -\hat{p}_{mk}log\hat{p}_{mk}$

Podobnie jak indeks Giniego przyjmuje małe wartości (bliskie zero) gdy $\hat{p}_{mk}$ jest bliskie 0 lub 1.

-----
##### *Zadanie 4*

Przygotuj zbiór danych `pima` do dalszej pracy. Podziel na zbiór zmiennych objaśniających i zmienną celu (utwórz zmienne `X` i `y`), a następnie podziel je na zbiór treningowy i testowy (`X_train`, `X_test`, `y_train` i `y_test`) w proporcji 7:3.


-----
##### *Zadanie 5*

Zbuduj model drzewa dla danych z Zadania 4. Zapisz dopasowany model w zmiennej `Tree`.


In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Tree =


Po dopasowaniu modelu możemy obejrzeć strukturę drzewa, wyznaczyć predykcje modelu oraz policzyć dokładność (*accuracy*) na zbiorze testowym.

In [ ]:
# Wizualizacja drzewa zbudowanego w Zadaniu 5
tree.plot_tree(
    Tree,
    feature_names=X_train.columns,
    class_names=[str(label) for label in Tree.classes_],
    filled=True,
)
plt.show()


In [ ]:
# Predykcje dla 10 pierwszych obserwacji w zbiorze testowym
X_test_sample = X_test.iloc[:10]
X_test_sample

In [ ]:
real_classes = y_test.iloc[:10]
predicted_classes = Tree.predict(X_test_sample)
predicted_probabilities = Tree.predict_proba(X_test_sample)

pd.DataFrame({
    "Real Class": real_classes,
    "Predicted Class": predicted_classes,
    "Predicted Probabilities": list(predicted_probabilities)
})

In [ ]:
# Dla klasyfikatora metoda score zwraca accuracy.
print(f"Accuracy na zbiorze testowym: {Tree.score(X_test, y_test):.3f}")

#### Macierz pomyłek i metryki klasyfikacji

W zadaniu klasyfikacji nie wystarczy wiedzieć, ile predykcji było poprawnych. Warto też sprawdzić, jakiego rodzaju błędy popełnia model. Macierz pomyłek zestawia klasy rzeczywiste z klasami przewidzianymi przez model.

- **True Positive (TP):** model przewidział klasę pozytywną i była to prawda;
- **True Negative (TN):** model przewidział klasę negatywną i była to prawda;
- **False Positive (FP):** model przewidział klasę pozytywną, ale była to pomyłka ("fałszywy alarm");
- **False Negative (FN):** model przewidział klasę negatywną, ale była to pomyłka ("przeoczenie")

<img src="confusion_matrix.png" width="400"/>

Na podstawie macierzy pomyłek obliczamy między innymi:

- **Dokładność (accuracy):** odsetek wszystkich poprawnych predykcji,
  $$\mathrm{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}.$$
- **Czułość (recall):** odsetek rzeczywistych pozytywów poprawnie rozpoznanych przez model,
  $$\mathrm{Recall} = \frac{TP}{TP + FN}.$$
- **Precyzję (precision):** odsetek trafnych predykcji pozytywnych,
  $$\mathrm{Precision} = \frac{TP}{TP + FP}.$$
- **Miara F1:** średnia harmoniczna precyzji i czułości,
  $$\mathrm{F1} = 2 \cdot \frac{\mathrm{Precision} \cdot \mathrm{Recall}}{\mathrm{Precision} + \mathrm{Recall}}.$$



-----
##### *Zadanie 6*

Wyznacz macierz pomyłek oraz miary `accuracy`, `precision`, `recall` i `f1` dla predykcji drzewa z Zadania 5. Porównaj macierz pomyłek dla zbioru testowego z macierzą dla zbioru treningowego.
